# PFE Spark — Entraînement de la résolution (modèle hybride)

Entraîne la **régression logistique hybride** décrite dans le rapport pour prédire la **résolution** des tickets Apache Spark.

Approche : embeddings sémantiques `all-mpnet-base-v2` (768 d) sur `text_for_res`, **concaténés** aux caractéristiques tabulaires de `MARTS_ML.MART_ML`, puis `LogisticRegression(class_weight='balanced', C=3.0)`.

Ce notebook est l'équivalent reproductible de `load/train_save_resolution_model.py` (qui produit le `.pkl` déployé et `results/evaluation_report.txt`).

> **Note de cohérence rapport ↔ code (17 vs 19 features).**
> Ce notebook entraîne le modèle **tel qu'il a réellement tourné** : embeddings (768 d) + **19** caractéristiques tabulaires (incluant `resolution_days` et `n_resolution_changes`), soit un vecteur de 787 dimensions, d'où les métriques **81,3 % d'accuracy / 26,9 % de macro-F1**.
> Le **rapport documente 17 caractéristiques** : il exclut volontairement de la *spécification* ces deux variables, identifiées comme une **fuite directe de la cible** (elles ne sont de toute façon jamais disponibles au triage d'un nouveau ticket). L'écart 17 (rapport) vs 19 (code) est donc **assumé et documenté** ; un ré-entraînement strict à 17 features constitue l'étape suivante si l'on souhaite des métriques 17-features.

In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
from sentence_transformers import SentenceTransformer  # ajouter 'sentence-transformers' aux packages du notebook
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import json

session = get_active_session()
print('Session Snowflake active :', session.get_current_database())

## 1. Chargement de MART_ML (texte + 19 features tabulaires)

In [ ]:
# 19 caractéristiques tabulaires (telles qu'utilisées par train_save_resolution_model.py).
# resolution_days et n_resolution_changes sont incluses ici (modèle tel qu'il a tourné) ;
# le rapport documente les 17 features hors fuite de cible — voir note ci-dessus.
TABULAR_FEATURES = [
    'n_total_changes', 'n_status_changes', 'n_priority_changes',
    'n_assignee_changes', 'n_resolution_changes', 'was_escalated',
    'n_people_involved', 'n_links_total', 'n_duplicates', 'n_blocks',
    'n_blocked_by', 'n_relates', 'n_comments', 'n_commenters',
    'resolution_days', 'summary_length', 'description_length',
    'n_container', 'has_parent',
]
feat_str = ', '.join(TABULAR_FEATURES)

df = session.sql(f"""
    SELECT key, split, issuetype, resolution, text_for_res, {feat_str}
    FROM PFE_SPARK.MARTS_ML.MART_ML
    WHERE split IN ('train', 'validation')
    ORDER BY key
""").to_pandas()
df.columns = [c.lower() for c in df.columns]
print(f'Chargé {len(df):,} tickets')
print(df['split'].value_counts())

## 2. Branche textuelle — embeddings all-mpnet-base-v2 (768 d)

In [ ]:
train = df[df['split'] == 'train'].reset_index(drop=True)
val   = df[df['split'] == 'validation'].reset_index(drop=True)

model = SentenceTransformer('all-mpnet-base-v2')

def embed(texts):
    return model.encode(texts, batch_size=256, show_progress_bar=True,
                        normalize_embeddings=True)

train_emb_res = embed(train['text_for_res'].fillna('').tolist())
val_emb_res   = embed(val['text_for_res'].fillna('').tolist())
print('Embeddings :', train_emb_res.shape, val_emb_res.shape)

## 3. Branche tabulaire + concaténation (vecteur 768 + 19 = 787 d)

In [ ]:
scaler    = StandardScaler()
train_tab = scaler.fit_transform(train[TABULAR_FEATURES].fillna(0).clip(upper=1e6))
val_tab   = scaler.transform(val[TABULAR_FEATURES].fillna(0).clip(upper=1e6))

X_train_res = np.hstack([train_emb_res, train_tab])
X_val_res   = np.hstack([val_emb_res,   val_tab])
print('X_train_res :', X_train_res.shape, '| X_val_res :', X_val_res.shape)

## 4. Entraînement LogisticRegression (résolution)

In [ ]:
clf_res = LogisticRegression(
    class_weight='balanced', max_iter=1000, C=3.0, solver='lbfgs', n_jobs=-1
)
clf_res.fit(X_train_res, train['resolution'])

pred = clf_res.predict(X_val_res)
f1   = f1_score(val['resolution'], pred, average='macro', zero_division=0)
acc  = (pred == val['resolution'].values).mean()
print(f'Résolution — accuracy={acc:.4f}  macro-F1={f1:.4f}')
print(classification_report(val['resolution'], pred, zero_division=0))

## 5. Sauvegarde dans le Snowflake Model Registry

Le modèle journalisé est le classifieur hybride (entrée = embedding 768 d concaténé aux 19 features tabulaires standardisées). En déploiement local, `load/train_save_resolution_model.py` produit en outre `clf_resolution.pkl`, `scaler.pkl` et `meta.json`.

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session)
reg.log_model(
    clf_res,
    model_name='resolution_classifier',
    version_name='v2_hybrid_mpnet_768_plus_19',
    conda_dependencies=['scikit-learn', 'pandas', 'numpy'],
    comment=f'LR balanced C=3.0, embeddings all-mpnet-base-v2 (768d) + 19 features tabulaires. accuracy={acc:.4f} macro-F1={f1:.4f}'
)
print('Modèle hybride sauvegardé dans le Model Registry.')